CoPilot cannot generate image descriptions using just the IIIF manifest URLs; instead, it requires images to be embedded in the spreadsheet.

In [2]:
import pandas as pd
import openpyxl
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter
import requests
from io import BytesIO
from PIL import Image as PILImage
import os

# Load the consolidated TSV
consolidated_file = '/Users/maeve/Documents/DAIMS-search/consolidated_exhibition_data.tsv'
df = pd.read_csv(consolidated_file, sep='\t', encoding='latin-1')

# Display columns and sample data
print("Columns:", df.columns.tolist())
print("\nDataset keys:", df['dataset_key'].unique() if 'dataset_key' in df.columns else "N/A")
print("\nFirst few rows:")
print(df.head())

Columns: ['manifest', 'record_url', 'dataset_key', 'dataset_name', 'geometry_type', 'coordinates', 'thumbnail']

Dataset keys: <StringArray>
['handmade', 'custom_iiif', 'comp1']
Length: 3, dtype: str

First few rows:
                                            manifest  \
0  https://iiif.library.leeds.ac.uk/presentation/...   
1  https://iiif.library.leeds.ac.uk/presentation/...   
2  https://iiif.library.leeds.ac.uk/presentation/...   
3  https://iiif.library.leeds.ac.uk/presentation/...   
4  https://iiif.library.leeds.ac.uk/presentation/...   

                                   record_url dataset_key  \
0  https://digital.library.leeds.ac.uk/15933/    handmade   
1  https://digital.library.leeds.ac.uk/15941/    handmade   
2  https://digital.library.leeds.ac.uk/15944/    handmade   
3  https://digital.library.leeds.ac.uk/15930/    handmade   
4  https://digital.library.leeds.ac.uk/15925/    handmade   

             dataset_name geometry_type  \
0  Hand-mapped Leeds Data       Poly

In [3]:
# Filter for hand-mapped data (dataset_key == 'handmade')
hand_mapped = df[df['dataset_key'] == 'handmade'].copy()
hand_mapped.columns

Index(['manifest', 'record_url', 'dataset_key', 'dataset_name',
       'geometry_type', 'coordinates', 'thumbnail'],
      dtype='str')

In [ ]:
def download_and_resize_image(url, max_width=150, max_height=200):
    """Download image from URL and resize for Excel embedding"""
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            img = PILImage.open(BytesIO(response.content))
            
            # Resize while maintaining aspect ratio
            img.thumbnail((max_width, max_height), PILImage.Resampling.LANCZOS)
            
            # Save to BytesIO for Excel embedding
            img_io = BytesIO()
            img.save(img_io, format='PNG')
            img_io.seek(0)
            return img_io
    except Exception as e:
        print(f"  Error downloading {url}: {e}")
    return None

def extract_iiif_metadata(manifest_url):
    """Extract metadata from IIIF manifest"""
    metadata = {
        'iiif_label': None,
        'iiif_extent': None,
        'iiif_date': None,
        'iiif_collections': None
    }
    
    if not manifest_url or pd.isna(manifest_url):
        return metadata
    
    try:
        response = requests.get(manifest_url, timeout=10)
        if response.status_code != 200:
            return metadata
        
        manifest = response.json()
        
        # Extract label
        if 'label' in manifest:
            label = manifest['label']
            if isinstance(label, dict):
                if 'en' in label and isinstance(label['en'], list):
                    metadata['iiif_label'] = label['en'][0]
                elif 'none' in label and isinstance(label['none'], list):
                    metadata['iiif_label'] = label['none'][0]
            elif isinstance(label, str):
                metadata['iiif_label'] = label
        
        # Extract metadata from metadata array (label/value pairs)
        if 'metadata' in manifest:
            for item in manifest['metadata']:
                if isinstance(item, dict):
                    label_text = item.get('label')
                    value_text = item.get('value')
                    
                    # Normalize label to string
                    if isinstance(label_text, dict):
                        if 'en' in label_text:
                            label_text = label_text['en'][0] if isinstance(label_text['en'], list) else label_text['en']
                        elif 'none' in label_text:
                            label_text = label_text['none'][0] if isinstance(label_text['none'], list) else label_text['none']
                    
                    # Normalize value to string
                    if isinstance(value_text, dict):
                        if 'en' in value_text:
                            value_text = value_text['en'][0] if isinstance(value_text['en'], list) else value_text['en']
                        elif 'none' in value_text:
                            value_text = value_text['none'][0] if isinstance(value_text['none'], list) else value_text['none']
                    
                    # Map to our target fields
                    if label_text:
                        label_lower = str(label_text).lower()
                        if 'extent' in label_lower and not metadata['iiif_extent']:
                            metadata['iiif_extent'] = value_text
                        elif 'date' in label_lower and not metadata['iiif_date']:
                            metadata['iiif_date'] = value_text
                        elif 'collection' in label_lower and not metadata['iiif_collections']:
                            metadata['iiif_collections'] = value_text
        
        return metadata
        
    except Exception as e:
        print(f"  Error fetching IIIF metadata from {manifest_url}: {e}")
        return metadata

# Create Excel workbook
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Hand-Mapped Items"

# Get all columns from hand_mapped, add Image column at start, and new IIIF metadata columns
tsv_columns = hand_mapped.columns.tolist()
iiif_columns = ['iiif_label', 'iiif_extent', 'iiif_date', 'iiif_collections']
all_columns = ['image'] + tsv_columns + iiif_columns
ws.append(all_columns)

# Style header row
header_fill = openpyxl.styles.PatternFill(start_color="CCCCCC", end_color="CCCCCC", fill_type="solid")
header_font = openpyxl.styles.Font(bold=True)
for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font

# Set column widths
ws.column_dimensions['A'].width = 20  # Image column
for col_idx in range(2, len(all_columns) + 1):
    col_letter = get_column_letter(col_idx)
    ws.column_dimensions[col_letter].width = 15

# Populate rows with data, images, and IIIF metadata
print("Embedding images and extracting IIIF metadata...")
for idx, (_, row) in enumerate(hand_mapped.iterrows(), start=2):
    ws.row_dimensions[idx].height = 150  # Height for images
    
    # Add all TSV data columns (starting from column B)
    for col_idx, col_name in enumerate(tsv_columns, start=2):
        col_letter = get_column_letter(col_idx)
        ws[f'{col_letter}{idx}'] = row[col_name]
    
    # Extract and add IIIF metadata
    manifest_url = row.get('manifest')
    print(f"  Row {idx}: Processing manifest {str(manifest_url)[:50]}...")
    iiif_metadata = extract_iiif_metadata(manifest_url)
    
    # Add IIIF metadata columns
    for col_idx, col_name in enumerate(iiif_columns, start=len(tsv_columns) + 2):
        col_letter = get_column_letter(col_idx)
        ws[f'{col_letter}{idx}'] = iiif_metadata[col_name]
    
    # Download and embed image in column A
    thumbnail_url = row.get('thumbnail')
    if thumbnail_url and pd.notna(thumbnail_url):
        print(f"    Downloading image {str(thumbnail_url)[:50]}...")
        img_io = download_and_resize_image(thumbnail_url)
        if img_io:
            try:
                xl_img = XLImage(img_io)
                ws.add_image(xl_img, f'A{idx}')
                print(f"Image embedded")
            except Exception as e:
                print(f"Failed to embed: {e}")
        else:
            print(f"Image unavailable")
    else:
        print(f"No thumbnail URL")

# Save the workbook
output_path = '/Users/maeve/Documents/DAIMS-search/handmade_exhibition_with_images_new.xlsx'
wb.save(output_path)
print(f"\n✓ Excel file saved to: {output_path}")
print(f"  Total items: {len(hand_mapped)}")
print(f"  Columns: {', '.join(all_columns)}")


Embedding images and extracting IIIF metadata...
  Row 2: Processing manifest https://iiif.library.leeds.ac.uk/presentation/cc/j...
    ✓ Image embedded
  Row 3: Processing manifest https://iiif.library.leeds.ac.uk/presentation/cc/h...
    ✓ Image embedded
  Row 4: Processing manifest https://iiif.library.leeds.ac.uk/presentation/cc/n...
    ✓ Image embedded
  Row 5: Processing manifest https://iiif.library.leeds.ac.uk/presentation/cc/x...
    ✓ Image embedded
  Row 6: Processing manifest https://iiif.library.leeds.ac.uk/presentation/cc/r...
    ✓ Image embedded
  Row 7: Processing manifest https://iiif.library.leeds.ac.uk/presentation/cc/n...
    ✓ Image embedded
  Row 8: Processing manifest https://iiif.library.leeds.ac.uk/presentation/cc/v...
    ✓ Image embedded
  Row 9: Processing manifest https://iiif.library.leeds.ac.uk/presentation/cc/g...
    ✓ Image embedded
  Row 10: Processing manifest https://iiif.library.leeds.ac.uk/presentation/cc/j...
    ✓ Image embedded
  Row 11: Proc